In [ ]:
import pandas as pd, io

# PASUL 0 — Citirea datelor din fișierul local
cadru_date_firme = pd.read_csv("lab5.csv", dtype={'caem': str})

cadru_date_firme['caem'] = cadru_date_firme['caem'].str.zfill(4)
cadru_date_firme['diviziune'] = cadru_date_firme['caem'].str[:2].astype(int)

# PASUL 1 — Secțiune din diviziune, coduri invalide, structură
intervale_sectiuni = [('A',1,3),('B',5,9),('C',10,33),('D',35,35),('E',36,39),
                      ('F',41,43),('G',45,47),('H',49,53),('I',55,56),('J',58,63),
                      ('K',64,66),('L',68,68),('M',69,75),('N',77,82),('O',84,84),
                      ('P',85,85),('Q',86,88),('R',90,93),('S',94,96),('T',97,98),
                      ('U',99,99)]

def determina_sectiunea(diviziune: int):
    for litera_sectiune, limita_inferioara, limita_superioara in intervale_sectiuni:
        if limita_inferioara <= diviziune <= limita_superioara:
            return litera_sectiune
    return None

cadru_date_firme['sectiune'] = cadru_date_firme['diviziune'].map(determina_sectiunea)
coduri_invalide = cadru_date_firme[cadru_date_firme['sectiune'].isna()]
coduri_valide   = cadru_date_firme.dropna(subset=['sectiune'])

sinteza_pe_sectiuni = coduri_valide.groupby('sectiune').agg(
    firme=('id', 'count'),
    angajati=('angajati', 'sum'),
)
sinteza_pe_sectiuni['pondere_%'] = (
    100 * sinteza_pe_sectiuni['angajati'] / sinteza_pe_sectiuni['angajati'].sum()
).round(1)

print("=" * 55)
print("PAS 1 — Structura pe secțiuni")
print("=" * 55)
print(sinteza_pe_sectiuni)
print("\nCoduri invalide:")
print(coduri_invalide[['id', 'denumire', 'caem']].to_string(index=False))

# PASUL 2 — Crosswalk 1:n cu alocare
tabel_corespondenta = pd.DataFrame({
    'cod_intern': ['R1','R2','R3','R4','R4'],
    'nace':       ['10.71','56.10','47.11','62.01','62.02'],
    'pondere':    [1, 1, 1, 0.6, 0.4],
})

registru_firme = pd.DataFrame({
    'firma':      list('ABCDEF'),
    'cod_intern': ['R1','R2','R3','R4','R4','R4'],
    'angajati':   [40, 30, 100, 50, 20, 10],
})

alocare_combinata = registru_firme.merge(tabel_corespondenta, on='cod_intern')
alocare_combinata['angajati_alocati'] = (
    alocare_combinata['angajati'] * alocare_combinata['pondere']
)

print("\n" + "=" * 55)
print("PAS 2 — Crosswalk cu alocare")
print("=" * 55)
print(alocare_combinata.groupby('nace')['angajati_alocati'].sum())

tip_asociere = tabel_corespondenta.groupby('cod_intern')['nace'].nunique()
print(f"\nTipuri: 1:1 = {(tip_asociere == 1).sum()}, "
      f"1:n = {(tip_asociere > 1).sum()} (R4)")
print(f"Control: {registru_firme['angajati'].sum()} = "
      f"{alocare_combinata['angajati_alocati'].sum():.0f}")

# PASUL 3 — Indicatori de calitate
publicatii = pd.DataFrame({
    'lucrare':         [f'L{i}' for i in range(1, 13)],
    'intarziere_zile': [0,0,0,2,0,0,5,0,0,0,1,0],
})

procent_punctualitate = 100 * (publicatii['intarziere_zile'] == 0).mean()
rata_raspuns          = 100 * 900 / 2000
completitudine        = 100 * coduri_valide.shape[0] / len(cadru_date_firme)

print("\n" + "=" * 55)
print("PAS 3 — Indicatori de calitate")
print("=" * 55)
print(f"Punctualitate:   {procent_punctualitate:.1f}%")
print(f"Rata de răspuns: {rata_raspuns:.1f}%")
print(f"Completitudine:  {completitudine:.1f}%")

# EXERCIȚIUL 1 — Firme cu cod 0300 și 9900
print("\n" + "=" * 55)
print("EXERCIȚIUL 1 — Firme cu cod 0300 și 9900")
print("=" * 55)

cadru_date_firme_cu_coduri_test = pd.concat([
    cadru_date_firme,
    pd.DataFrame({
        'id':         [17, 18],
        'denumire':   ['Firma 0300', 'Firma 9900'],
        'activitate': ['test', 'test'],
        'caem':       ['0300', '9900'],
        'angajati':   [10, 20],
    }),
], ignore_index=True)

cadru_date_firme_cu_coduri_test['caem'] = (
    cadru_date_firme_cu_coduri_test['caem'].str.zfill(4)
)
cadru_date_firme_cu_coduri_test['diviziune'] = (
    cadru_date_firme_cu_coduri_test['caem'].str[:2].astype(int)
)
cadru_date_firme_cu_coduri_test['sectiune'] = (
    cadru_date_firme_cu_coduri_test['diviziune'].map(determina_sectiunea)
)

print(cadru_date_firme_cu_coduri_test[
    cadru_date_firme_cu_coduri_test['id'].isin([17, 18])
][['id','caem','diviziune','sectiune']].to_string(index=False))

# EXERCIȚIUL 2 — Ponderi R4: 0.7 / 0.3
print("\n" + "=" * 55)
print("EXERCIȚIUL 2 — Ponderi R4: 0.7 / 0.3")
print("=" * 55)

tabel_corespondenta_alternativ = tabel_corespondenta.copy()
tabel_corespondenta_alternativ.loc[
    (tabel_corespondenta_alternativ['cod_intern'] == 'R4') &
    (tabel_corespondenta_alternativ['nace'] == '62.01'), 'pondere'
] = 0.7
tabel_corespondenta_alternativ.loc[
    (tabel_corespondenta_alternativ['cod_intern'] == 'R4') &
    (tabel_corespondenta_alternativ['nace'] == '62.02'), 'pondere'
] = 0.3

alocare_combinata_alternativa = registru_firme.merge(
    tabel_corespondenta_alternativ, on='cod_intern'
)
alocare_combinata_alternativa['angajati_alocati'] = (
    alocare_combinata_alternativa['angajati']
    * alocare_combinata_alternativa['pondere']
)

print(alocare_combinata_alternativa.groupby('nace')['angajati_alocati'].sum())
print(f"\nControl: {registru_firme['angajati'].sum()} = "
      f"{alocare_combinata_alternativa['angajati_alocati'].sum():.0f}")

# EXERCIȚIUL 3 — Punctualitate cu toleranță ≤ 1 zi
print("\n" + "=" * 55)
print("EXERCIȚIUL 3 — Punctualitate (≤ 1 zi = la termen)")
print("=" * 55)

procent_punctualitate_cu_toleranta = 100 * (
    publicatii['intarziere_zile'] <= 1
).mean()
print(f"Punctualitate (≤ 1 zi): {procent_punctualitate_cu_toleranta:.1f}%")
print(f"(față de {procent_punctualitate:.1f}% fără toleranță)")

# EXERCIȚIUL 4 — Principii CoP vizate de fiecare indicator
print("\n" + "=" * 55)
print("EXERCIȚIUL 4 — Principii CoP și măsuri propuse")
print("=" * 55)
print("""
1. Punctualitate (75.0%) — principiul 13 CoP (actualitate/timeliness)
   Măsură: notificări automate cu 3 zile înainte de termen.

2. Rata de răspuns (45.0%) — principiile 12 (acuratețe) și 9 (relevanță)
   Măsură: simplificarea chestionarului + canal online de colectare.

3. Completitudine cod CAEM (93.8%) — principiile 12 (acuratețe)
   și 14 (completitudine)
   Măsură: validare automată a codului la introducere + listă de sugestii.
""")

PAS 1 — Structura pe secțiuni
          firme  angajati  pondere_%
sectiune                            
A             2        30        3.0
C             2       107       10.7
F             1       150       14.9
G             1       120       12.0
H             1        35        3.5
I             2        82        8.2
J             2       107       10.7
M             1         6        0.6
P             1        48        4.8
Q             2       319       31.8

Coduri invalide:
 id denumire caem
 16  Firma X 4400

PAS 2 — Crosswalk cu alocare
nace
10.71     40.0
47.11    100.0
56.10     30.0
62.01     48.0
62.02     32.0
Name: ang_alocati, dtype: float64

Tipuri: 1:1 = 3, 1:n = 1 (R4)
Control: 250 = 250

PAS 3 — Indicatori de calitate
Punctualitate:   75.0%
Rata de răspuns: 45.0%
Completitudine:  93.8%

EXERCIȚIUL 1 — Firme cu cod 0300 și 9900
 id caem  diviziune sectiune
 17 0300          3        A
 18 9900         99        U

EXERCIȚIUL 2 — Ponderi R4: 0.7 / 0.3
nace
10.71